# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Maryam-Yaqoob/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

The queue: score every (client, content) pair with the trained model (`w05_model.ipynb`'s
Logistic Regression, retrained on the full honest feature set), sort descending by predicted
probability of `is_declining_next_half`, and attach a plain-language reason code built from
*which* feature pushed the score up — not just the number.

| Reason code | Trigger | Action |
|---|---|---|
| `low_ctr_for_position` | `ctr_first_half` well below its `position_tier`'s weighted average | Review title/meta description — visibility is fine, clicks aren't |
| `thin_coverage` | `active_days_first_half` < 5 | Low confidence flag — the model's read on this page is based on very little data; verify manually before acting |
| `high_volume_at_risk` | High `impressions_first_half` AND flagged | Priority review — most impressions to lose if the decline is real |
| `low_signal` | Flagged, but no single feature stands out | Lower-confidence flag — worth a quick look, not urgent |


In [ ]:
# ---- Setup: same DuckDB + HF pattern as earlier notebooks. Run in Colab with your HF_TOKEN. ----
%pip -q install duckdb scikit-learn
import duckdb, pandas as pd, numpy as np, os

from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"
FACT = f"read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet')"
COLS = {"impressions": "gsc_impressions", "clicks": "gsc_clicks", "position": "gsc_avg_position"}

feature_frame = con.sql(f"""
    WITH first_half AS (
        SELECT client_hash_id, content_hash_id,
               SUM({COLS['impressions']}) AS impressions_first_half,
               SUM({COLS['clicks']})      AS clicks_first_half,
               AVG(CASE WHEN {COLS['impressions']} > 0 THEN {COLS['position']} END) AS avg_position_first_half,
               COUNT(DISTINCT CASE WHEN {COLS['impressions']} > 0 THEN report_date END) AS active_days_first_half
        FROM {FACT}
        WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
        GROUP BY 1, 2
    ),
    second_half AS (
        SELECT client_hash_id, content_hash_id,
               SUM({COLS['impressions']}) AS impressions_second_half
        FROM {FACT}
        WHERE report_date BETWEEN DATE '2026-03-16' AND DATE '2026-03-31'
        GROUP BY 1, 2
    )
    SELECT f.*, s.impressions_second_half,
           f.clicks_first_half * 100.0 / NULLIF(f.impressions_first_half, 0) AS ctr_first_half,
           CASE WHEN s.impressions_second_half < 0.8 * f.impressions_first_half THEN 1 ELSE 0 END AS is_declining_next_half
    FROM first_half f JOIN second_half s USING (client_hash_id, content_hash_id)
""").df()
print(f"Feature frame: {len(feature_frame):,} rows")

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

HONEST_FEATURES = ["impressions_first_half", "clicks_first_half", "ctr_first_half",
                    "avg_position_first_half", "active_days_first_half"]

df = feature_frame.dropna(subset=HONEST_FEATURES).copy()
y = df["is_declining_next_half"]

train_clients, test_clients = train_test_split(
    df["client_hash_id"].unique(), test_size=0.3, random_state=42)
train_mask = df["client_hash_id"].isin(train_clients)

lr = LogisticRegression(max_iter=1000).fit(df.loc[train_mask, HONEST_FEATURES], y[train_mask])
df["predicted_prob"] = lr.predict_proba(df[HONEST_FEATURES])[:, 1]

def position_tier(p):
    if pd.isna(p) or p <= 0: return "no_data"
    if p <= 3: return "top_3"
    if p <= 10: return "page_1"
    if p <= 20: return "striking"
    if p <= 50: return "page_3_5"
    return "deep"

df["position_tier"] = df["avg_position_first_half"].apply(position_tier)
tier_ctr = df.groupby("position_tier").apply(
    lambda g: 100 * g["clicks_first_half"].sum() / g["impressions_first_half"].sum())
df["expected_ctr_for_tier"] = df["position_tier"].map(tier_ctr)

FLAG_THRESHOLD = 0.5  # predicted_prob cutoff -- tune against your own precision@K curve

def reason_code(row):
    if row["predicted_prob"] < FLAG_THRESHOLD:
        return "none"
    if row["active_days_first_half"] < 5:
        return "thin_coverage"
    if row["ctr_first_half"] < row["expected_ctr_for_tier"] * 0.7:
        return "low_ctr_for_position"
    if row["impressions_first_half"] > df["impressions_first_half"].quantile(0.75):
        return "high_volume_at_risk"
    return "low_signal"

df["reason_code"] = df.apply(reason_code, axis=1)
ranked = df.sort_values("predicted_prob", ascending=False).reset_index(drop=True)
print(ranked["reason_code"].value_counts())
ranked.head(10)[["client_hash_id", "content_hash_id", "predicted_prob", "reason_code"]]


## 2. Intended use and limits

**Who uses this:** a content/SEO lead deciding which pages to review this sprint, working from a
fixed review budget (e.g. the top 50-100 rows of the queue).

**Where it stops being valid:**
- Trained and validated on **March 2026** for the clients in this warehouse slice — not validated
  for other months, seasons, or clients outside this dataset.
- Predicts a **20%-drop-vs-first-half** outcome specifically — not "will this page ever decline,"
  not a general health score, not a ranking-position forecast.
- `thin_coverage`-flagged rows (fewer than 5 active days in the feature window) are lower-confidence
  by design — the model saw very little data for those pairs.
- This is decision-support, not an autopilot: it tells a human where to look first, not what to
  change or guarantees an outcome from acting.


In [ ]:
print("Valid for: month=2026-03, this warehouse release, is_declining_next_half target only.")
print("Not valid for: other months without retraining, other prediction targets, unattended automation.")
print(f"\nRows flagged thin_coverage (lowest confidence): {(ranked['reason_code']=='thin_coverage').sum():,}")


## 3. Human review + the no-go list

**What a person must check before acting on a flagged page:** the actual page in a browser (does
it still match the query it targets?), whether a recent site-wide change (redesign, migration,
tracking change) could explain a drop better than content quality, and whether the flagged
decline lines up with a known seasonal dip for that client's industry.

**What should never be automated directly from this queue:** deleting or de-indexing a page,
changing URLs/redirects, or any action based on a single `thin_coverage`-flagged row without a
human first checking the underlying daily data — low-coverage estimates are exactly where this
model is most likely to be wrong.


In [ ]:
no_go_list = [
    "Never auto-delete or de-index a flagged page",
    "Never auto-redirect/restructure URLs from this queue alone",
    "Never act on a thin_coverage row without checking daily-level data first",
    "Never treat a flag as proof the CONTENT is the cause (site-wide changes can mimic this)",
]
for item in no_go_list:
    print("-", item)


## 4. Monitoring / retrain triggers

- **Base-rate drift:** if `is_declining_next_half`'s true rate in a new month drifts far from this
  month's 32.7%, the model's calibration is stale — retrain rather than reuse.
- **Precision@10 drop on fresh data:** if a new month's actual Precision@10 falls well below the
  ~50% floor the rule already clears, the model isn't earning its complexity anymore.
- **Warehouse schema changes:** if `fact_content_daily_performance`'s column names or grain shift
  (new GSC/GA4 fields, a new partition scheme), the feature-build query needs re-verification
  before trusting any new output.
- **New client cohorts:** clients added after this training month may have different baseline
  patterns entirely — check performance separately for any client not in the original 55.


In [ ]:
triggers = {
    "base_rate_drift": "new month's actual decline rate moves far from 32.7%",
    "precision_drop": "Precision@10 on fresh month falls below the ~50% rule floor",
    "schema_change": "column names/grain in fact_content_daily_performance change",
    "new_clients": "clients outside the original 55 need separate validation before trusting flags",
}
for k, v in triggers.items():
    print(f"{k}: {v}")


## 5. Exports for the paper

Writing the ranked queue and a small metrics receipt to `work/outputs/` — the paper's Results and
Recommendations sections read from these files, not from re-running the whole pipeline inline.


In [ ]:
os.makedirs("work/outputs", exist_ok=True)

output_cols = ["client_hash_id", "content_hash_id", "impressions_first_half", "clicks_first_half",
               "ctr_first_half", "avg_position_first_half", "position_tier",
               "active_days_first_half", "predicted_prob", "reason_code"]
ranked[output_cols].to_csv("work/outputs/action_playbook_queue.csv", index=False)
print(f"Wrote work/outputs/action_playbook_queue.csv -- {len(ranked):,} rows")

import json as _json
playbook_metrics = {
    "month": MONTH,
    "n_rows": int(len(ranked)),
    "n_flagged": int((ranked["reason_code"] != "none").sum()),
    "reason_code_counts": ranked["reason_code"].value_counts().to_dict(),
    "flag_threshold": FLAG_THRESHOLD,
}
with open("work/outputs/action_playbook_metrics.json", "w") as f:
    _json.dump(playbook_metrics, f, indent=2)
print("Wrote work/outputs/action_playbook_metrics.json")
print(playbook_metrics)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.